In [ ]:
import os
import geopandas as gpd

gdf = gpd.read_file(r"D:\Datasets\Imagenes Satelitales\Estados Unidos\NY State Imagery\NYC\boro_bronx_sp06\lot6_nyc_liz_06.shp")
gdf.crs

In [ ]:
gdf#.to_crs("EPSG:4326")

In [ ]:
import os
import zipfile
import glob
from pathlib import Path
import geopandas as gpd
import rioxarray
from osgeo import gdal
from tqdm import tqdm

def process_ny_imagery(folder_path, shapefile_name):
    """
    1. Reads the Shapefile to find image names.
    2. Unzips the corresponding .jp2 files.
    3. Creates a VRT mosaic.
    4. Returns a lazy-loaded Xarray object.
    """
    base_dir = Path(folder_path)
    shp_path = base_dir / shapefile_name
    
    # Define where to put extracted images (can be a subfolder to keep things clean)
    extract_dir = base_dir / "extracted_jp2"
    extract_dir.mkdir(exist_ok=True)
    
    # --- Step 1: Read the Shapefile Index ---
    print(f"📖 Reading index: {shp_path}")
    try:
        gdf = gpd.read_file(shp_path)
        # Get the list of expected .jp2 filenames from the 'IMAGE' column
        image_list = gdf['IMAGE'].tolist()
        print(f"   Found {len(image_list)} images in the index.")
    except Exception as e:
        print(f"❌ Error reading Shapefile: {e}")
        return None

    # --- Step 2: Unzip the specific files ---
    print("🚀 Extracting files...")
    
    extracted_files = []
    
    # We loop through the list from the shapefile to ensure we match the data
    for image_name in tqdm(image_list, desc="Unzipping"):
        # Construct the zip filename. Assumption: "000262.jp2" is inside "000262.zip"
        zip_name = image_name.replace('.jp2', '.zip') 
        zip_path = base_dir / zip_name
        
        target_jp2 = extract_dir / image_name
        
        # Track this path for the VRT later
        extracted_files.append(str(target_jp2))

        # Skip if already extracted
        if target_jp2.exists():
            continue
            
        if not zip_path.exists():
            # Try finding the zip with different casing or prefix if strict matching fails
            # (Optional safeguard, skipping for now based on your description)
            print(f"⚠️ Warning: Zip file not found for {image_name}")
            continue

        try:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                # We only want the .jp2 and the .j2w (world file)
                # The .j2w is crucial for the grid alignment!
                files_to_extract = [f for f in zf.namelist() if f.lower().endswith(('.jp2', '.j2w', '.aux'))]
                
                for file in files_to_extract:
                    # Extract to the specific folder
                    zf.extract(file, extract_dir)
        except zipfile.BadZipFile:
            print(f"❌ Corrupt zip file: {zip_path}")

    # --- Step 3: Create the Virtual Mosaic (VRT) ---
    print("🗺️  Building VRT Mosaic...")
    vrt_path = base_dir / "mosaic.vrt"
    
    # Filter list to ensure files actually exist (in case of unzip errors)
    valid_files = [f for f in extracted_files if os.path.exists(f)]
    
    if not valid_files:
        print("❌ No valid images found to create mosaic.")
        return None

    # Build VRT
    # srcNodata=0 ensures that the empty space (since shape is not square) 
    # is treated as transparent/nodata rather than black.
    vrt_options = gdal.BuildVRTOptions(resampleAlg='nearest', addAlpha=False, srcNodata=0)
    gdal.BuildVRT(str(vrt_path), valid_files, options=vrt_options)
    print(f"✅ VRT created at: {vrt_path}")

    # --- Step 4: Open as Xarray ---
    print("📡 Loading into Xarray (Lazy)...")
    
    # chunks is mandatory for lazy loading
    da = rioxarray.open_rasterio(vrt_path, chunks={'x': 2048, 'y': 2048})
    
    # --- Handle the 4-Band Issue (Infrared) ---
    if da.rio.count == 4:
        print("⚠️ 4 Bands detected. Dropping Band 4 (Infrared) to keep RGB.")
        da = da.sel(band=[1, 2, 3])
        
    print(f"✅ Final Data Shape: {da.shape}")
    return da

# --- Usage ---
# folder = r"C:\Your\Path\To\Data"
# shp_name = "your_index_file.shp"
# xarray_data = process_ny_imagery(folder, shp_name)

In [ ]:
gdf[gdf["COUNTY"]=="Bronx"]

In [ ]:
files = os.listdir(r"D:\Datasets\Imagenes Satelitales\Estados Unidos\NY State Imagery\NYC")
files = [f for f in files if ".zip" in f]
files

borough_names = ["bronx", "brooklyn", "manhattan", "queens", "staten_island"]
# Create the dictionary
borough_dict = {}

for boro in borough_names:
    borough_dict[boro] = []
    for filename in files:
        if boro in filename:
            borough_dict[boro].append(filename)

borough_dict

In [ ]:
borough_translation = {
    'bronx': 'Bronx',
    'manhattan':'New York',
    'staten_island': 'Richmond',
    'brooklyn': 'Kings',
    'queens': 'Queens',
}

In [ ]:
import os
import zipfile
import glob
from pathlib import Path
import geopandas as gpd
import rioxarray
# from osgeo import gdal
from tqdm import tqdm

for boro, f_list in borough_dict.items():

    for file in f_list:
        base_dir = rf"D:\Datasets\Imagenes Satelitales\Estados Unidos\NY State Imagery\NYC\{file}"
        shp_path = rf"{base_dir}\lot6_nyc_liz_06.shp"

        # Define where to put extracted images (can be a subfolder to keep things clean)
        extract_dir = rf"{base_dir}\extracted_jp2"
        os.makedirs(extract_dir, exist_ok=True)

        # --- Step 1: Read the Shapefile Index ---
        print(f"📖 Reading index: {shp_path}")
        try:
            gdf = gpd.read_file(shp_path)
            # Get the list of expected .jp2 filenames from the 'IMAGE' column
            image_list = gdf[gdf["COUNTY"]=="Bronx"]['IMAGE'].tolist()
            print(f"   Found {len(image_list)} images in the index.")

        except Exception as e:
            raise ValueError(f"❌ Error reading Shapefile: {e}")

        # --- Step 2: Unzip the specific files ---
        print("🚀 Extracting files...")

        extracted_files = []

        # We loop through the list from the shapefile to ensure we match the data
        for image_name in tqdm(image_list, desc="Unzipping"):
            # Construct the zip filename. Assumption: "000262.jp2" is inside "000262.zip"
            zip_name = image_name.replace('.jp2', '.zip') 
            zip_path = rf"{base_dir}\{zip_name}"
            
            target_jp2 = rf"{extract_dir}\{image_name}"
            
            # Track this path for the VRT later
            extracted_files.append(str(target_jp2))

            # Skip if already extracted
            if os.path.exists(target_jp2):
                continue
                
            if not os.path.exists(zip_path):
                # Try finding the zip with different casing or prefix if strict matching fails
                # (Optional safeguard, skipping for now based on your description)
                print(f"⚠️ Warning: Zip file not found for {image_name}")
                print(zip_name, zip_path)

            try:
                with zipfile.ZipFile(zip_path, 'r') as zf:
                    # We only want the .jp2 and the .j2w (world file)
                    # The .j2w is crucial for the grid alignment!
                    files_to_extract = [f for f in zf.namelist() if f.lower().endswith(('.jp2', '.j2w', '.aux'))]
                    
                    for file in files_to_extract:
                        # Extract to the specific folder
                        zf.extract(file, extract_dir)

            except zipfile.BadZipFile:
                print(f"❌ Corrupt zip file: {zip_path}")



In [ ]:
import os
import dask
import zipfile
import geopandas as gpd
import rioxarray
import xarray as xr
from pathlib import Path
from tqdm import tqdm
import shutil
import xarray as xr
import rioxarray
import numpy as np
from pathlib import Path
from dask.diagnostics import ProgressBar



def move_images(image_list, folder, output_folder):
    ''' All the images have the same 2006 shapefile. So we can move them all to a yearly folder and work with that! '''   
   
    # Add year to output_folder
    year = folder.name.split("_")[-1][-2:]
    output_folder = output_folder / year
    os.makedirs(output_folder, exist_ok=True)

    count = 0
    for jp2_name in tqdm(image_list, desc="Prep"):

        zip_name = jp2_name.replace(".jp2", ".zip")
        j2w_name = jp2_name.replace(".jp2", ".j2w")
        aux_name = jp2_name.replace(".jp2", ".aux")
        tab_name = jp2_name.replace(".jp2", ".tab")

        jp2_path = folder / jp2_name
        zip_path = folder / zip_name

        # If already extracted
        if jp2_path.exists():
            
            # Move to the output_folder
            for f in [jp2_name, j2w_name, aux_name, tab_name]:

                if (folder / f).exists():
                    shutil.move(folder / f, output_folder / f)

            count += 1
            continue
            
        # Unzip if missing
        elif zip_path.exists():
            try:
                with zipfile.ZipFile(zip_path, 'r') as zf:
                    # We need the jp2 and the world file (.j2w)
                    files = [f for f in zf.namelist() if f.lower().endswith(('.jp2', '.j2w', '.aux', '.tab'))]
                    for f in files:
                        zf.extract(f, output_folder / f)

                count += 1

            except Exception as e:
                print(f"❌ Error unzipping {zip_path}: {e}")

        else:
            continue

    if count == 0:
        print(f"❌ No images found in {folder}.")
        return
    

def load_mosaic(images_path, reference_zarr_path):
    import concurrent.futures
    import rioxarray
    import xarray as xr
    from pathlib import Path
    from tqdm import tqdm
    import win32file

    print("Setting windows max file opening number...")
    print(f"Old limit: {win32file._getmaxstdio()}")
    win32file._setmaxstdio(8192)
    print(f"New limit: {win32file._getmaxstdio()}")

    os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR" # Don't scan directory contents
    os.environ["GDAL_NUM_THREADS"] = "1" # Disable GDAL's internal threading (let Dask handle it)
    os.environ["OPJ_NUM_THREADS"] = "1"  # Disable OpenJPEG threading
    os.environ["GDAL_CACHEMAX"] = "512"  # Increase cache size


    # 1. LOAD THE MASTER GRID
    # We only load the coordinates (lightweight), not the data.
    print(f"📏 Loading reference grid from: {reference_zarr_path}")
    ref_ds = xr.open_zarr(reference_zarr_path)
    
    # Load X and Y into memory as pure Numpy arrays for fast lookup
    # (Assuming strict sorting, which Zarr enforces)
    ref_x = ref_ds.x.values
    ref_y = ref_ds.y.values
    
    # Get resolution (assuming constant) to help with matching
    dx = ref_x[1] - ref_x[0]
    dy = ref_y[1] - ref_y[0]

    base_dir = Path(images_path)
    assert base_dir.exists(), f"Directory not found: {base_dir}"

    images = list(base_dir.glob("*.jp2"))
    if not images:
        print("❌ No .jp2 files found.")
        return None

    print(f"📡 Loading {len(images)} tiles into Xarray (Parallel Lazy)...")

    # --- Worker Function for Parallel Execution ---
    def open_single_image(p):
        try:
            # chunks={} activates Dask (lazy loading) - crucial!
            ds = rioxarray.open_rasterio(p, chunks={"x": 2500, "y": 2500})

            # --- THE "SNAP" TRICK ---
            # 1. Find where this image starts in the Reference X array
            #    We use searchsorted for speed.
            x_start_val = ds.x.values[0]
            y_start_val = ds.y.values[0] # Be careful with Y direction!

            # Find closest index in Reference
            # (We use 'abs' to find nearest because floating points might be slightly off)
            ix = (np.abs(ref_x - x_start_val)).argmin()
            iy = (np.abs(ref_y - y_start_val)).argmin()

            # 2. Slice the Reference Coordinates
            #    We take exactly the number of pixels this image has
            n_rows = ds.rio.height
            n_cols = ds.rio.width
            
            # Handle Y direction (Satellite images often have negative dy, meaning Y decreases)
            # If dy is negative, the "start" is actually the top, so we slice downwards?
            # Easiest way: Slice based on the found index and check length.
            
            # Safety check: ensure we don't go out of bounds
            if ix + n_cols > len(ref_x) or iy + n_rows > len(ref_y):
                    # Fallback for edge cases (rare): just round the original coords
                    ds.coords['x'] = ds.coords['x'].round(1)
                    ds.coords['y'] = ds.coords['y'].round(1)
            else:
                # Overwrite the image coords with the PERFECT reference coords
                # Note: This assumes standard top-left origin. 
                # If Y is descending (standard map), we might need to adjust the slice direction.
                # But typically, if you match the start index, you just take the slice:
                
                # Check direction of Y in reference
                if ref_y[0] > ref_y[-1]: # Descending Y (Standard for Northern Hemisphere)
                        # For descending, the 'start' value of the image is the MAX Y.
                        # So we slice from iy to iy + n_rows
                        new_y = ref_y[iy : iy + n_rows]
                else:
                        new_y = ref_y[iy : iy + n_rows]

                new_x = ref_x[ix : ix + n_cols]

                # Assign the clean coordinates
                ds = ds.assign_coords(x=new_x, y=new_y)

            # ------------------------
            ds = ds.rio.write_nodata(0, inplace=False)
            ds = ds.astype("uint8")
            ds.coords['x'] = ds.coords['x'].round(1)
            ds.coords['y'] = ds.coords['y'].round(1)
            return ds
        except Exception as e:
            print(f"⚠️ Fail: {p.name} - {e}")
            return None

    datasets = []
    
    # --- Execute in Parallel ---
    # Max workers = 16 or 32 is usually the sweet spot for file headers.
    # Your Gen 5 SSD can handle way more, but Python overhead limits us.
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
        # map returns an iterator, so we wrap it in list() to force execution
        results = list(tqdm(executor.map(open_single_image, images), total=len(images), desc="Opening Headers"))

    # Remove failures (None)
    datasets = [ds for ds in results if ds is not None]

    # --- Step 3: Combine ---
    print("🧩 Stitching mosaic together (combine_by_coords)...")
    try:
        # This part is pure CPU (sorting coordinates) and cannot be easily parallelized
        mosaic = xr.combine_by_coords(datasets)
        print(f"✅ Mosaic Complete! Shape: {mosaic.shape}")
        return mosaic
    except ValueError as e:
        print(f"❌ Merge failed: {e}")
        return None
        
def convert_to_zarr(folder_path, output_file, reference_zarr, client=None):
    """
    Converts the slow JP2 mosaic into a fast, chunked Zarr dataset.
    """
    from numcodecs import Blosc, Delta

    folder_path = Path(folder_path)
    output_file = Path(output_file)

    # Create the Lazy Virtual Mosaic
    print("⏳ Building virtual layer from JP2s...")
    mosaic = load_mosaic(folder_path, reference_zarr)

    if mosaic is None:
        print(f"❌ Zarr creation failed: {folder_path}")

        return

    mosaic = mosaic.astype("uint8")

    # We remove the _FillValue from the attributes so it doesn't conflict 
    # with the encoding settings during export.
    if "_FillValue" in mosaic.attrs:
        del mosaic.attrs["_FillValue"]
    

    # Ensure standard names for coordinates so tools generally recognize it
    if 'band' in mosaic.coords:
        mosaic.coords['band'] = np.array([1, 2, 3, 4]) # RGB-NIR
    
    mosaic.name = "value"
    
    print("✂️ Masking last 2 bits for higher compression...")
    mosaic = mosaic & 0xFC    

    # Export to Zarr
    # This is the heavy lifting step. It will read the JP2s and write the Zarr.
    print(f"💾 Exporting to {output_file}...")
    print("   (This will take a while, but only needs to be done once!)")

    # Clear any previous encoding info carried over from the JP2s
    mosaic.encoding = {}
    filters = [Delta(dtype="uint8")]
    compressor = Blosc(cname='zstd', clevel=5, shuffle=Blosc.BITSHUFFLE)
    mosaic.to_zarr(
        output_file, 
        mode='w', 
        compute=True,
        encoding={"value": {
            "dtype": "uint8",
            "compressor": compressor,
            "filters": filters,
            "_FillValue": 0
        }},
        write_empty_chunks=False,
        zarr_format=2 
    )

    print("✅ Done! Data is now in Zarr format.")
    return

# # --- Usage ---
# folder = r"D:\Datasets\Imagenes Satelitales\Estados Unidos\NY State Imagery\NYC\boro_bronx_sp10"
# shp = "lot6_nyc_liz_06.shp"

# # Run the conversion
# zarr_path = convert_to_zarr(folder, shp)


# Transform the raw folder of images to zarr
years = [
    # '08',
    '16',
    '10',
    '12',
    # '14',
    '18',
    '20',
    '22',
    '24',
]

from IPython.display import display
from dask.distributed import Client, LocalCluster

reference_zarr = r"E:\Datasets\Imagenes Satelitales\New York City\nyc_2014.zarr"

# Configure the cluster explicitly for CPU-bound image work
# n_workers=8: Matches your physical cores (adjust based on your CPU)
# threads_per_worker=1: Ensures no GIL contention. 
# memory_limit='4GB': Prevents workers from eating all RAM.
cluster = LocalCluster(n_workers=3, threads_per_worker=1, memory_limit='9GB')
client = Client(cluster)
display(client)

for year in years:
    print("Process will be run on Dask client:")

    # Move from HDD (D) to Super Fast SSD (E)
    folder_path = rf"E:\Datasets\Imagenes Satelitales\New York City\NYC Images\{year}"
    output_file= rf"E:\Datasets\Imagenes Satelitales\New York City\nyc_20{year}.zarr"
    mosaic = convert_to_zarr(
        folder_path, 
        output_file,
        reference_zarr,
        client
    )


In [ ]:
# Trim test polygon based on NY buildings so no black images are generated
import geopandas as gpd

buildings = gpd.read_file(r"/mnt/c/Working Papers/NY State Aerial Imagery Prototype/ny_state_aerial_imagery_prototype/data/raw/USBuildingFootprints/NewYork.geojson/NewYork.geojson")
test_poly = gpd.read_parquet(r"/mnt/c/Working Papers/NY State Aerial Imagery Prototype/ny_state_aerial_imagery_prototype/data/raw/Test_NYC_Area.parquet")


DataSourceError: '/mnt/c/Working Papers/NY State Aerial Imagery Prototype/ny_state_aerial_imagery_prototype/data/raw/Test_NYC_Area.parquet' not recognized as being in a supported file format.; It might help to specify the correct driver explicitly by prefixing the file path with '<DRIVER>:', e.g. 'CSV:path'.

In [3]:
import xarray as xr
import rioxarray

def create_lowres_overview(zarr_path, output_tif_path, target_mb=500):
    # 1. Open Zarr lazily (loading metadata only)
    # mask_and_scale=False keeps it as uint8 (fast & low memory)
    ds = xr.open_zarr(zarr_path, mask_and_scale=False)
    
    # Identify the spatial variables (usually 'band_data', 'red', etc.)
    # If your zarr has a main variable like 'image' or 'band_data', extract it.
    # Otherwise, xarray usually treats the whole dataset as the object.
    # We assume 'ds' behaves like a DataArray or has one main variable.
    if isinstance(ds, xr.Dataset):
        # Taking the first variable found that has spatial dims
        var_name = list(ds.data_vars)[0] 
        da = ds[var_name]
    else:
        da = ds

    # 2. Calculate the "Step" (Decimation Factor)
    # Current size in Bytes (approximate)
    current_bytes = da.nbytes
    target_bytes = target_mb * 1024 * 1024
    
    ratio = current_bytes / target_bytes
    # Since area is x * y, the step for one dimension is sqrt(ratio)
    # We add a buffer (+2) to ensure we stay under the limit
    step = int(ratio ** 0.5) + 2
    
    print(f"Original size: {current_bytes / 1e9:.2f} GB")
    print(f"Calculated step: Taking 1 out of every {step} pixels.")

    # 3. Slice (Subsample)
    # This is INSTANT because it's lazy. It only decides WHICH chunks to read.
    # We use .isel() to select indices by step
    # Adjust 'x' and 'y' if your dims are named 'lon'/'lat'
    if 'x' in da.dims and 'y' in da.dims:
        subset = da.isel(x=slice(None, None, step), y=slice(None, None, step))
    elif 'lon' in da.dims and 'lat' in da.dims:
        subset = da.isel(lon=slice(None, None, step), lat=slice(None, None, step))
    else:
        raise ValueError(f"Could not find x/y or lon/lat dims. Found: {da.dims}")

    # 4. Write to GeoTIFF
    # We use LZW compression to make it even smaller
    print("Writing to TIF... (this triggers the download/read)")
    subset.rio.to_raster(
        output_tif_path, 
        compress="LZW", 
        tiled=True, 
        dtype="uint8" # Ensure it writes as lightweight byte data
    )
    print(f"Done! Saved to {output_tif_path}")

# --- Usage ---
year = 18  # example
zarr_path = rf"E:/Datasets/Imagenes Satelitales/New York City/nyc_20{year}.zarr"
create_lowres_overview(zarr_path, f"nyc_overview_20{year}.tif", target_mb=500)

Original size: 416.00 GB
Calculated step: Taking 1 out of every 30 pixels.
Writing to TIF... (this triggers the download/read)
Done! Saved to nyc_overview_2018.tif


In [ ]:
test_poly.dissolve()

,geometry
0,"MULTIPOLYGON (((1016464.974 135043.777, 996878..."


In [ ]:
test_poly

# Test for checking if the zarr/xarray are good!

In [ ]:
folder = r"E:\Datasets\Imagenes Satelitales\New York City\NYC Images\24"
# mosaic = load_mosaic(folder)
mosaic = load_mosaic_parallel(folder)


In [ ]:
from dask.distributed import Client
from IPython.display import display

client = Client()
display(client)

mosaic.to_zarr(r"E:\Datasets\Imagenes Satelitales\New York City\test.zarr", mode="w")

In [ ]:
import matplotlib.pyplot as plt
for year in years:
    mosaic = xr.open_zarr(rf"E:\Datasets\Imagenes Satelitales\New York City\nyc_20{year}.zarr")#.load()
    image = mosaic.sel(x=slice(1025500, 1026500), y=slice(272000, 271000)).load()
    import earthpy.plot as ep
    fig, ax = plt.subplots(figsize=(10,10))
    ep.plot_rgb(image["value"].to_numpy(), rgb=(0,1,2), ax=ax)
    plt.title(f"NYC Aerial Imagery - 20{year}")
    plt.savefig(f"E:/Datasets/Imagenes Satelitales/New York City/nyc_20{year}.png")

In [7]:
import xarray as xr
year = "16"
mosaic = xr.open_dataset(rf"/mnt/e/Datasets/Imagenes Satelitales/New York City/nyc_20{year}.zarr",engine="zarr", mask_and_scale=False)#.load()


In [8]:
mosaic

<xarray.Dataset> Size: 416GB
Dimensions:      (band: 4, y: 320000, x: 325000)
Coordinates:
  * band         (band) int64 32B 1 2 3 4
    spatial_ref  int64 8B ...
  * x            (x) float64 3MB 9.075e+05 9.075e+05 ... 1.07e+06 1.07e+06
  * y            (y) float64 3MB 2.775e+05 2.775e+05 ... 1.175e+05 1.175e+05
Data variables:
    value        (band, y, x) uint8 416GB ...